# 📓 Notebook 01 — Web Application Offensive

SQLi, XSS, SSRF, IDOR, JWT, auth bypass, LFI/RFI, deserialization, API abuse.
Targets: Juice Shop, DVWA, WebGoat, bWAPP, crAPI, VAmPI.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE = '/content/drive/MyDrive/redteam'
os.makedirs(CACHE, exist_ok=True)
print("cache:", CACHE)

In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

In [ ]:
%%bash
# ---- pick ONE, uncomment ----
# 24 GB L4 (likely your card) — best agent brain for the size
ollama pull hf.co/bartowski/Qwen3-Coder-30B-A3B-Instruct-abliterated-GGUF:Q4_K_M
# 16 GB T4:
# ollama pull gpt-oss:20b
# 40 GB A100:
# ollama pull hf.co/bartowski/Llama-3.3-70B-Instruct-abliterated-GGUF:Q4_K_M
# 80-96 GB H100 / G4:
# ollama pull hf.co/bartowski/gpt-oss-120b-GGUF:MXFP4
ollama list

In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
cd /content/hexstrike
[ -f install.sh ] && bash install.sh >/var/log/hexstrike_install.log 2>&1 || true
python3 -m venv .venv && .venv/bin/pip -q install -r requirements.txt
nohup .venv/bin/python hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
for i in $(seq 1 60); do curl -sf localhost:8888/health >/dev/null && break; sleep 3; done
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity
docker exec redteam bash -lc 'apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y kali-tools-top10 metasploit-framework >/dev/null 2>&1; echo toolbox-ready' 

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
sleep 15
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1

In [ ]:
%%bash
# --- install Go for ProjectDiscovery tooling ---
if ! command -v go >/dev/null; then
  curl -fsSL https://go.dev/dl/go1.23.5.linux-amd64.tar.gz -o /tmp/go.tgz
  rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
  echo 'export PATH=$PATH:/usr/local/go/bin' >> /etc/profile.d/go.sh
fi
export PATH=$PATH:/usr/local/go/bin
go version

In [ ]:
%%bash
export PATH=$PATH:/usr/local/go/bin:/root/go/bin
go install -v github.com/projectdiscovery/nuclei/v3/cmd/nuclei@latest
go install -v github.com/projectdiscovery/httpx/cmd/httpx@latest
go install -v github.com/projectdiscovery/subfinder/v2/cmd/subfinder@latest
go install -v github.com/projectdiscovery/katana/cmd/katana@latest
go install -v github.com/ffuf/ffuf/v2@latest
go install -v github.com/hahwul/dalfox/v2@latest
/root/go/bin/nuclei -update-templates -silent

In [ ]:
%%bash
apt-get -qq install -y sqlmap nikto whatweb wafw00f gobuster feroxbuster
pip -q install arjun commix 2>/dev/null || true
git clone --depth 1 https://github.com/ticarpi/jwt_tool /content/jwt_tool
git clone --depth 1 https://github.com/s0md3v/Arjun /content/arjun 2>/dev/null || true
docker pull -q zaproxy/zap-stable
docker run -d --name zap --network host zaproxy/zap-stable zap.sh -daemon -host 0.0.0.0 -port 8090 -config api.disablekey=true
echo "tools ready" 

In [ ]:
%%bash
# --- spin the labs (all --network host, distinct ports) ---
docker run -d --name juice   --network host bkimminich/juice-shop
docker run -d --name dvwa    --network host vulnerables/web-dvwa
docker run -d --name webgoat --network host webgoat/webgoat
docker run -d --name bwap    --network host raesene/bwapp
docker run -d --name vampi   --network host erev0s/vampi
sleep 30
for p in 3000 80 8080; do curl -s -o /dev/null -w "port $p -> %{http_code}\n" http://localhost:$p; done

In [ ]:
# --- one-shot recon sweep the agent can riff on ---
target = "http://localhost:3000"   # Juice Shop
!httpx -u {target} -title -tech-detect -status-code -silent
!whatweb {target}
!wafw00f {target}
!nuclei -u {target} -severity low,medium,high,critical -silent | head -40